# B5 · Mô hình nhìn ảnh + trả lời bằng chữ (Qwen2.5-VL-3B) — KHÔNG cần học

Dùng khi đề yêu cầu **sinh chữ** từ ảnh: hỏi đáp về ảnh (VQA), mô tả ảnh, đọc chữ trong ảnh, hoặc khi có quá ít dữ liệu để học.
Cần tải mô hình ~7,5 GB (B0 mục 5), VRAM ~8 GB. Hiểu được tiếng Việt.
⚠️ Phần này chưa chạy thử được trước (máy nhà không có GPU) — lỗi thì hỏi AI của BTC theo mẫu A.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())          # thư mục notebooks (chứa b2lib.py)
import numpy as np, pandas as pd
import b2lib as L
print("thư mục làm việc:", L.WORK)

In [ ]:
df = L.doc_bang_chuan()
itr, ite = (df.split == "train").to_numpy(), (df.split == "test").to_numpy()
y = df.loc[itr, "label"].astype(str).to_numpy()
ids_tr, ids_te = df.loc[itr, "id"].to_numpy(), df.loc[ite, "id"].to_numpy()
print(itr.sum(), "train |", ite.sum(), "test |", len(set(y)), "lớp")

## Thử trên 5 ảnh train trước, so với nhãn đúng

In [ ]:
tr5 = df[itr].head(5)
CAU_HOI = "Mô tả ngắn gọn bức ảnh này bằng tiếng Việt."     # ✏️ hoặc dùng tr5.text.tolist() nếu cột text là câu hỏi
tl = L.vlm_tra_loi(tr5.image, CAU_HOI, max_new_tokens=48, he_thong="Trả lời ngắn gọn bằng tiếng Việt.")
display(pd.DataFrame({"text": tr5.text, "label": tr5.label, "may_tra_loi": tl}))

## Chạy cho cả test → file nộp

In [ ]:
te_df = df[ite]
cau_hoi = te_df.text.tolist() if (te_df.text != "").all() else CAU_HOI
tra_loi = L.vlm_tra_loi(te_df.image, cau_hoi, max_new_tokens=32, he_thong="Trả lời ngắn gọn bằng tiếng Việt.")
FILE_MAU = ""; COT_ID, COT_NHAN = "id", "answer"      # ✏️
L.ghi_nop(ids_te, tra_loi, None, FILE_MAU or None, COT_ID, COT_NHAN, ten="nop_vlm")

**Mẹo:** đề là phân loại nhưng ít dữ liệu → hỏi `"Ảnh thuộc loại nào trong: A, B, C? Chỉ trả lời đúng một từ."` rồi so chuỗi trả lời với danh sách nhãn.